# nuPlan Paper Experiments

这个 notebook 专门服务论文产出，不负责跑底层 benchmark。

结构按三块组织：

1. `Zero-shot / Transfer`
2. `Fine-tune / In-domain`
3. `Ablation`

你可以把不同来源的结果 CSV 合并进同一张表：

- 我们自己的 `FM+CBF / Pure FM / IDM-proxy`
- 官方 `IDM / PDM-Closed`
- 外部 `DiffusionPlanner / FlowPlanner`

建议论文写法：

- 把 `Training regime` 明确标成 `zero_shot / fine_tuned / in_domain / no_learning`
- `DiffusionPlanner / FlowPlanner` 当前默认视为 `in_domain` baseline
- zero-shot 表的重点是迁移性和安全性，不是绝对公平的 fully-trained SOTA 对比

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path('/new_world/cockatiel/ra').resolve()
PROJECT_ROOT = ROOT / 'safeflow-nuplan-transfer' / 'dmpc_fm_cbf'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dmpc_fm_cbf.paper_experiment_reporting import (
    attach_method_metadata,
    build_finetune_table,
    build_method_metadata,
    build_zero_shot_table,
    export_latex_table,
    export_markdown_table,
    load_external_baseline_csv,
    load_metrics_csv,
    make_figure_paths,
    merge_metrics_tables,
    plot_ablation,
    plot_regime_comparison,
    plot_zero_shot_comparison,
    summarize_metrics,
)

PAPER_OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'cache' / 'paper_experiments'
PAPER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_PATHS = make_figure_paths(PAPER_OUTPUT_DIR)
PAPER_OUTPUT_DIR


In [ ]:
# ------------------------------
# Result file registry
# ------------------------------
# 你后面只需要改这些路径。

ZERO_SHOT_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'nuplan_transfer' / 'metrics.csv'
FINE_TUNE_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'nuplan_transfer_finetuned' / 'metrics.csv'

# 外部 baseline CSV：不存在也没关系，notebook 会跳过。
OFFICIAL_IDM_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'external_baselines' / 'official_idm_metrics.csv'
OFFICIAL_PDM_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'external_baselines' / 'official_pdm_closed_metrics.csv'
DIFFUSION_PLANNER_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'external_baselines' / 'diffusion_planner_metrics.csv'
FLOW_PLANNER_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'external_baselines' / 'flow_planner_metrics.csv'

# 消融 CSV：推荐你后面按这个 schema 保存
# required columns: paper_bucket, tau0 or K or road_margin_m, collision_rate, goal_reached_rate, road_boundary_violations, time_to_goal_s
ABLATION_TAU_CSV = PROJECT_ROOT / 'notebooks' / 'cache' / 'paper_experiments' / 'ablation_tau.csv'
ABLATION_K_CSV = PROJECT_ROOT / 'notebooks' / 'cache' / 'paper_experiments' / 'ablation_k.csv'
ABLATION_MARGIN_CSV = PROJECT_ROOT / 'notebooks' / 'cache' / 'paper_experiments' / 'ablation_margin.csv'

RESULT_PATHS = {
    'zero_shot': ZERO_SHOT_METRICS,
    'fine_tune': FINE_TUNE_METRICS,
    'official_idm': OFFICIAL_IDM_METRICS,
    'official_pdm_closed': OFFICIAL_PDM_METRICS,
    'diffusion_planner': DIFFUSION_PLANNER_METRICS,
    'flow_planner': FLOW_PLANNER_METRICS,
}
RESULT_PATHS


In [ ]:
tables = []

if ZERO_SHOT_METRICS.exists():
    zero_df = load_metrics_csv(ZERO_SHOT_METRICS)
    tables.append(zero_df)
    print('[OK] zero-shot metrics:', ZERO_SHOT_METRICS)
else:
    print('[MISS] zero-shot metrics:', ZERO_SHOT_METRICS)

if FINE_TUNE_METRICS.exists():
    ft_df = load_metrics_csv(
        FINE_TUNE_METRICS,
        method_overrides={
            'fm_cbf': 'fm_cbf_finetuned',
            'pure_fm': 'pure_fm_finetuned',
        },
    )
    tables.append(ft_df)
    print('[OK] fine-tune metrics:', FINE_TUNE_METRICS)
else:
    print('[MISS] fine-tune metrics:', FINE_TUNE_METRICS)

for method_key, path in [
    ('official_idm', OFFICIAL_IDM_METRICS),
    ('official_pdm_closed', OFFICIAL_PDM_METRICS),
    ('diffusion_planner', DIFFUSION_PLANNER_METRICS),
    ('flow_planner', FLOW_PLANNER_METRICS),
]:
    if path.exists():
        ext_df = load_external_baseline_csv(path)
        if 'method' not in ext_df.columns or ext_df['method'].nunique() != 1:
            ext_df['method'] = method_key
        tables.append(ext_df)
        print(f'[OK] external baseline {method_key}:', path)
    else:
        print(f'[MISS] external baseline {method_key}:', path)

if not tables:
    raise RuntimeError('No metrics table found. Run benchmark first or place baseline CSVs in the configured paths.')

metrics_df = merge_metrics_tables(tables)
method_meta = build_method_metadata(metrics_df['method'].unique())
metrics_enriched = attach_method_metadata(metrics_df, method_meta)
summary_df = summarize_metrics(metrics_enriched)

summary_csv = PAPER_OUTPUT_DIR / 'summary_metrics.csv'
summary_df.to_csv(summary_csv, index=False)
print('saved:', summary_csv)
display(summary_df.head(20))


## Zero-shot Table

这张表建议放：

- `SafeFlow (FM+CBF)`
- `Pure FM`
- `IDM / PDM-Closed`
- `DiffusionPlanner / FlowPlanner`

但要保留 `training_regime` 一列。

In [ ]:
zero_shot_table = build_zero_shot_table(summary_df)
display(zero_shot_table)

export_markdown_table(zero_shot_table, PAPER_OUTPUT_DIR / 'zero_shot_table.md')
export_latex_table(zero_shot_table, PAPER_OUTPUT_DIR / 'zero_shot_table.tex')
plot_zero_shot_comparison(summary_df, FIG_PATHS.zero_shot_barplot)
display(Image(filename=str(FIG_PATHS.zero_shot_barplot)))


## Fine-tune Table

这张表建议对比：

- `SafeFlow (Fine-tuned)`
- `Pure FM (Fine-tuned)`
- `IDM / PDM-Closed`
- `DiffusionPlanner / FlowPlanner`

如果你还没有 fine-tune 结果，这一节会先退化成基线占位。

In [ ]:
fine_tune_table = build_finetune_table(summary_df)
display(fine_tune_table)

export_markdown_table(fine_tune_table, PAPER_OUTPUT_DIR / 'fine_tune_table.md')
export_latex_table(fine_tune_table, PAPER_OUTPUT_DIR / 'fine_tune_table.tex')
plot_regime_comparison(summary_df, FIG_PATHS.regime_barplot)
display(Image(filename=str(FIG_PATHS.regime_barplot)))


## Ablation

推荐三组消融：

- `tau0` / `tau1` 门控
- `K` 候选轨迹数
- `road_margin_m`

CSV schema 建议：

- `paper_bucket`
- `tau0` 或 `K` 或 `road_margin_m`
- `collision_rate`
- `goal_reached_rate`
- `road_boundary_violations`
- `time_to_goal_s`

In [ ]:
if ABLATION_TAU_CSV.exists():
    tau_df = pd.read_csv(ABLATION_TAU_CSV)
    display(tau_df)
    plot_ablation(
        tau_df,
        x_col='tau0',
        metrics=[('collision_rate', 'Collision Rate ↓'), ('goal_reached_rate', 'Goal Reached Rate ↑')],
        title='CBF Gate Ablation',
        output_path=FIG_PATHS.ablation_tau,
    )
    display(Image(filename=str(FIG_PATHS.ablation_tau)))
else:
    print('[MISS] tau ablation csv:', ABLATION_TAU_CSV)

if ABLATION_K_CSV.exists():
    k_df = pd.read_csv(ABLATION_K_CSV)
    display(k_df)
    plot_ablation(
        k_df,
        x_col='K',
        metrics=[('goal_reached_rate', 'Goal Reached Rate ↑'), ('time_to_goal_s', 'Time to Goal ↓')],
        title='Candidate Count Ablation',
        output_path=FIG_PATHS.ablation_k,
    )
    display(Image(filename=str(FIG_PATHS.ablation_k)))
else:
    print('[MISS] K ablation csv:', ABLATION_K_CSV)

if ABLATION_MARGIN_CSV.exists():
    margin_df = pd.read_csv(ABLATION_MARGIN_CSV)
    display(margin_df)
    plot_ablation(
        margin_df,
        x_col='road_margin_m',
        metrics=[('road_boundary_violations', 'Road Violations ↓'), ('time_to_goal_s', 'Time to Goal ↓')],
        title='Road CBF Margin Ablation',
        output_path=FIG_PATHS.ablation_margin,
    )
    display(Image(filename=str(FIG_PATHS.ablation_margin)))
else:
    print('[MISS] road margin ablation csv:', ABLATION_MARGIN_CSV)


## Files Produced

这个 notebook 会输出：

- `summary_metrics.csv`
- `zero_shot_table.md/.tex`
- `fine_tune_table.md/.tex`
- `zero_shot_comparison.png`
- `regime_comparison.png`
- `ablation_tau.png`
- `ablation_k.png`
- `ablation_margin.png`

都在：

- `/new_world/cockatiel/ra/safeflow-nuplan-transfer/dmpc_fm_cbf/notebooks/cache/paper_experiments/`